In [ ]:
import marimo as mo
import os
from nexus.simulator.run_linked_sim import run_sim
from nexus.simulator.grn.grnSim import GRNSim
from nexus.simulator.spatial.spatialSim import SpatialSim
from nexus.simulator.utils.file_manager import _read_file
from hydra import initialize_config_dir, compose
from omegaconf import OmegaConf
import pprint
import yaml
import matplotlib.pyplot as plt

## Loading config

In [ ]:
def get_config(config_name="test_config"):
    conf_path = os.path.join(os.getcwd(), "configs")
    with initialize_config_dir(version_base=None, config_dir=conf_path):
        cfg = compose(config_name=config_name)
    return cfg

In [ ]:
cfg = get_config("config")

In [ ]:
print(OmegaConf.to_yaml(cfg=cfg.grn))

# Running GRN Sim

## RNA Sim using SERGIO input files

In [ ]:
grn_sim = GRNSim(cfg=cfg.grn)

In [ ]:
rna_conc, prot_conc = grn_sim.run_sim(10)

In [ ]:
print(
    rna_conc.shape
)  ## RNA concentration after running simulator (No. of RNAs, No. of Cells, 1)
print(
    prot_conc.shape
)  ## Protein concentration after running simulator (No. of Proteins, No. of Cells, 1)

In [ ]:
## Plot concentration of RNAs in cell 0
fig = plt.figure(figsize=(12, 6))
_ticks = list(range(rna_conc.shape[0]))
plt.plot(rna_conc[:, 0])
plt.ylabel("Conc.")
plt.xlabel("RNA Idx")
plt.title("Concentration of RNAs in cell 0")

## Running RNA and protein sim

Uses a custom config for defining the RNA and Protein parameters

In [ ]:
with open("sample_data/sample_network_1cell.yaml", "r") as file:
    node_set, edge_set = yaml.safe_load(file)
print("RNA and protein configuration")
pprint.pprint(node_set)

print("Network configuration")
pprint.pprint(edge_set)

In [ ]:
cfg2 = get_config()

In [ ]:
print(OmegaConf.to_yaml(cfg=cfg2.grn))

In [ ]:
grn_sim2 = GRNSim(cfg=cfg2.grn)

In [ ]:
rna_conc2, prot_conc2 = grn_sim2.run_sim(10)

In [ ]:
## Plot concentration of RNAs in cell 0
_fig = plt.figure(figsize=(12, 6))
_subplots = _fig.subfigures(1, 2)

ax1 = _subplots[0].subplots()
_ticks = list(range(rna_conc2.shape[0]))
ax1.plot(rna_conc2[:, 0])
ax1.set_ylabel("Conc.")
ax1.set_xlabel("RNA Idx")
ax1.set_title("Concentration of RNAs in cell 0")

ax2 = _subplots[1].subplots()
ax2.plot(prot_conc2[:, 0])
ax2.set_ylabel("Conc.")
ax2.set_xlabel("Prot Idx")
ax2.set_title("Concentration of Protein in cell 0")

# Spatial Sim

In [ ]:
cfg3 = get_config("test_config")
print(OmegaConf.to_yaml(cfg3.spatial_sim))

In [ ]:
spatial_sim = SpatialSim(cfg3.spatial_sim)

In [ ]:
spatial_sim.run_sim()

In [ ]:
plt.figure(figsize=(12, 6))
_data = spatial_sim.logger.retrieve_chem_data(field_id=1)
_chem_data = []
for _i in range(len(set(_data["chem"]))):
    _idx = _data["chem"] == _i
    _chem_data.append(_data["conc"][_idx])
    plt.plot(_data["conc"][_idx], label=f"Chem - {str(_i)}")
plt.legend()
plt.xlabel("Timestep")
plt.ylabel("Conc.")
plt.title("Concentration of chemicals in Field 1 across timesteps")

# Linked Sim - Running spatial and GRN sim together

In [ ]:
cfg4 = get_config("cell_field_test")

In [ ]:
run_sim(cfg4)

In [ ]:
field_conc = _read_file("outputs/field_concs.pkl")

In [ ]:
_field = 2
_chem_conc = [[], [], [], []]
for i in field_conc:
    for j in range(field_conc[0].shape[1]):
        _chem_conc[j].append(i[_field][j])
for _i in range(len(_chem_conc)):
    plt.plot(_chem_conc[_i], label=f"Chem - {_i}")
_ticks = list(range(0, len(field_conc), 2))
plt.legend()
plt.title(f"Conc. plot of chemicals in field {str(_field)}")
plt.xlabel("Timestep")
plt.ylabel("Conc")
plt.xticks(_ticks)
plt.gca()